In [26]:
import os
from pprint import pprint
import pandas as pd

data_path = "bori_raw_data"
all_files = os.listdir(data_path)
txt_files = [file for file in all_files if file.endswith(".txt")]


all_bori_data = []
def flatten_verse_collection(verse_collection):
    first_line = verse_collection[0]
    first_vid = first_line[0]
    if first_vid[-1] == ' ': #heading
        out_vid = first_vid[:8] + 'h'
    else:
        out_vid = first_vid[:8]
    verse_sans_list = [v[1] for v in verse_collection]
    out_sans = '। '.join(verse_sans_list) + '।'
    return [out_vid, out_sans]

book_id_dict = {}

print('------ Collecting all verses in one CSV file ------')
for bidx, txt_file in enumerate(txt_files):
    file_path = os.path.join(data_path, txt_file)
    output_file_name = ''
    output_data = []

    verse_collection = []
    with open(file_path, 'r') as file:
        for idx, line in enumerate(file):
            line = line.strip()
            if idx == 0:
                book_name = line.split(': ')[-1]
                print(f'Reading Book {str(int(bidx+1)).zfill(2)}: {book_name}')
                book_id_dict[f'{str(int(bidx+1)).zfill(2)}'] = book_name
                output_file_name = f'{book_name}.json'
            
            if not line.startswith('%'):
                vid = line[:9]
                if verse_collection:
                    if verse_collection[-1][0][:8] == vid[:8]:
                        verse_collection.append([vid, line[10:]])
                        # output_data.append(flatten_verse_collection(verse_collection))
                        # verse_collection = []                        
                    else:
                        output_data.append(flatten_verse_collection(verse_collection))
                        verse_collection = []
                        verse_collection.append([vid, line[10:]])
                else:
                    verse_collection.append([vid, line[10:]])
                    
        if verse_collection:
            output_data.append(flatten_verse_collection(verse_collection))
            verse_collection = []   
                
                   
    all_bori_data.extend(output_data)
        
all_bori_data_df = pd.DataFrame(all_bori_data, columns=['bori_id', 'sans'])
all_bori_data_df.to_csv('all_bori_data.csv', index=False)

------ Collecting all verses in one CSV file ------
Reading Book 01: Adiparvan
Reading Book 02: Sabhaparvan
Reading Book 03: Aranyakaparvan
Reading Book 04: Virataparvan
Reading Book 05: Udyogaparvan
Reading Book 06: Bhismaparvan
Reading Book 07: Dronaparvan
Reading Book 08: Karnaparvan
Reading Book 09: Salyaparvan
Reading Book 10: Sauptikaparvan
Reading Book 11: Striparvan
Reading Book 12: Santiparvan
Reading Book 13: Anusasanaparvan
Reading Book 14: Asvamedhikaparvan
Reading Book 15: Asramavasikaparvan
Reading Book 16: Mausalaparvan
Reading Book 17: Mahaprasthanikaparvan
Reading Book 18: Svargarohanaparvan


In [28]:
all_bori_gtrans_data_df = pd.read_csv('all_bori_gtrans_data.csv', dtype={'bori_id': object})

all_bori_gtrans_data_df['book_id'] = all_bori_gtrans_data_df['bori_id'].str.slice(0, 2)
all_bori_gtrans_data_df['chapter_id'] = all_bori_gtrans_data_df['bori_id'].str.slice(2, 5)
all_bori_gtrans_data_df['verse_id'] = all_bori_gtrans_data_df['bori_id'].str.slice(5, 8)

all_bori_gtrans_data_df = all_bori_gtrans_data_df[['bori_id', 'book_id', 'chapter_id', 'verse_id', 'sans', 'gtrans']]

all_bori_gtrans_data_df.head()

,bori_id,book_id,chapter_id,verse_id,sans,gtrans
0,01001000,01,001,000,नारायणं नमस्कृत्य नरं चैव नरोत्तमम्। देवीं सरस...,"Nārāyaṇa, bowing down, and the best of men...."
1,01001001,01,001,001,लोमहर्षणपुत्र उग्रश्रवाः सूतः पौराणिको नैमिषार...,"The son of Lomaharsana, Ugrasrava, Suta, the m..."
2,01001002,01,001,002,समासीनानभ्यगच्छद्ब्रह्मर्षीन्संशितव्रतान्। विन...,He approached the brahmarshis who were seated ...
3,01001003,01,001,003,तमाश्रममनुप्राप्तं नैमिषारण्यवासिनः। चित्राः श...,The inhabitants of the forest of Naimisha have...
4,01001004,01,001,004,अभिवाद्य मुनींस्तांस्तु सर्वानेव कृताञ्जलिः। अ...,He greeted the sages and offered them all his ...


In [29]:
# Create formatted jsons
import json

for book_id in all_bori_gtrans_data_df['book_id'].unique():
    # print(book_id, book_id_dict[book_id])
    book_name = book_id_dict[book_id]
    book_df = all_bori_gtrans_data_df[all_bori_gtrans_data_df['book_id'] == book_id]
    
    book_obj = {
        "book_name": f'{book_name}',
        "book_number": int(book_id),
        "num_chapters": 0,
        "chapters": [],
        "book_total_verses": 0
    }
    
    book_chapter_count = 0
    book_total_verses_count = 0
    book_chapters = []
    for chapter_id in book_df['chapter_id'].unique():
        book_chapter_count += 1
        chapter_df = book_df[book_df['chapter_id'] == chapter_id]
        chapter_name = f'Canto {chapter_id}'
        
        chapter_obj = {
            "chapter_id": chapter_id,
            "chapter_name": chapter_name,
            "verses": [],
            "chapter_total_verses": 0
        }
        chapter_total_verses = 0
        chapter_verses = []
        for v_ind, verse in chapter_df.iterrows():
            book_total_verses_count += 1
            chapter_total_verses += 1
            
            v_sans = verse['sans']
            v_sans_list = v_sans.split('।')[:-1]
            v_sans_list = [vl.strip() for vl in v_sans_list]
            v_gtrans = verse['gtrans']
            
            verse_obj = {
                "verse_id": verse['verse_id'],
                "verse_translation": {
                    'google_trans': v_gtrans
                },
                "verse_sans_lines": v_sans_list
            }
            chapter_verses.append(verse_obj)
        
        chapter_obj['verses'] = chapter_verses
        chapter_obj['chapter_total_verses'] = chapter_total_verses
        book_chapters.append(chapter_obj)
    
    book_obj['num_chapters'] = book_chapter_count
    book_obj['chapters'] = book_chapters
    book_obj['book_total_verses'] = book_total_verses_count
    
    
    book_savefile_name = os.path.join('formatted_json_data', f'{book_id}_{book_id_dict[book_id]}.json')
    with open(book_savefile_name, 'w', encoding='utf-8') as f:
        json.dump(book_obj, f, ensure_ascii=False, indent=4)
    print(f'Created {book_savefile_name}')
    

Created formatted_json_data\01_Adiparvan.json
Created formatted_json_data\02_Sabhaparvan.json
Created formatted_json_data\03_Aranyakaparvan.json
Created formatted_json_data\04_Virataparvan.json
Created formatted_json_data\05_Udyogaparvan.json
Created formatted_json_data\06_Bhismaparvan.json
Created formatted_json_data\07_Dronaparvan.json
Created formatted_json_data\08_Karnaparvan.json
Created formatted_json_data\09_Salyaparvan.json
Created formatted_json_data\10_Sauptikaparvan.json
Created formatted_json_data\11_Striparvan.json
Created formatted_json_data\12_Santiparvan.json
Created formatted_json_data\13_Anusasanaparvan.json
Created formatted_json_data\14_Asvamedhikaparvan.json
Created formatted_json_data\15_Asramavasikaparvan.json
Created formatted_json_data\16_Mausalaparvan.json
Created formatted_json_data\17_Mahaprasthanikaparvan.json
Created formatted_json_data\18_Svargarohanaparvan.json
